# VS Code + Continue Extension

Test the coding assistant using **VS Code** with the [Continue](https://continue.dev) extension connected to MaaS.

This notebook will:
1. Discover your MaaS endpoints dynamically
2. Verify connectivity
3. Generate the Continue configuration file
4. Guide you through testing with expected screenshots

**Prerequisites:** Phases 0–3 completed

## Step 1: Discover Endpoints

In [ ]:
import os
import subprocess
import json
import urllib.request
import ssl
from dotenv import load_dotenv

load_dotenv()

# Try .env first, then fall back to oc CLI
CLUSTER_DOMAIN = os.getenv("CLUSTER_DOMAIN")
if not CLUSTER_DOMAIN:
    result = subprocess.run(
        ["oc", "get", "ingresses.config.openshift.io", "cluster",
         "-o", "jsonpath={.spec.domain}"],
        capture_output=True, text=True
    )
    if result.returncode == 0 and result.stdout.strip():
        CLUSTER_DOMAIN = result.stdout.strip()
    else:
        raise RuntimeError(
            "Cannot discover cluster domain. "
            "Set CLUSTER_DOMAIN in .env or run 'oc login' first."
        )

MAAS_HOST = f"https://maas-api.{CLUSTER_DOMAIN}"

token_result = subprocess.run(["oc", "whoami", "-t"], capture_output=True, text=True)
OC_TOKEN = token_result.stdout.strip()

ctx = ssl.create_default_context()
ctx.check_hostname = False
ctx.verify_mode = ssl.CERT_NONE

# Get models
req = urllib.request.Request(
    f"{MAAS_HOST}/v1/models",
    headers={"Authorization": f"Bearer {OC_TOKEN}"}
)
try:
    with urllib.request.urlopen(req, context=ctx) as resp:
        models_data = json.loads(resp.read())
    MODEL_NAME = models_data["data"][0]["id"]
except Exception as e:
    MODEL_NAME = "MODEL_NOT_FOUND"

# Get MCP servers
routes_result = subprocess.run(
    ["oc", "get", "httproute", "-n", "mcp-servers",
     "-l", "maas.opendatahub.io/managed=true",
     "-o", "jsonpath={range .items[*]}{.metadata.name}\n{end}"],
    capture_output=True, text=True
)
mcp_servers = {}
for line in routes_result.stdout.strip().split("\n"):
    if line:
        name = line.replace("mcp-route-", "")
        mcp_servers[name] = f"{MAAS_HOST}/mcp/{name}/mcp"

print(f"MaaS Endpoint  : {MAAS_HOST}/v1")
print(f"Model          : {MODEL_NAME}")
print(f"MCP Servers    : {len(mcp_servers)}")
print(f"Source         : {'.env' if os.getenv('CLUSTER_DOMAIN') else 'oc CLI'}")
print()
for name, url in mcp_servers.items():
    print(f"  {name:25s} -> {url}")

## Step 2: Verify Connectivity

In [ ]:
print("Connectivity Check:")
print("=" * 60)

try:
    test_req = urllib.request.Request(
        f"{MAAS_HOST}/v1/models",
        headers={"Authorization": f"Bearer {OC_TOKEN}"}
    )
    with urllib.request.urlopen(test_req, context=ctx) as resp:
        print(f"  [OK] Model endpoint - HTTP {resp.status}")
except Exception as e:
    print(f"  [FAIL] Model endpoint - {e}")

for name, url in mcp_servers.items():
    try:
        mcp_req = urllib.request.Request(
            url, headers={"Authorization": f"Bearer {OC_TOKEN}"}
        )
        with urllib.request.urlopen(mcp_req, context=ctx, timeout=5) as resp:
            print(f"  [OK] MCP: {name} - HTTP {resp.status}")
    except Exception as e:
        err_msg = str(e)[:50]
        if "200" in err_msg or "405" in err_msg:
            print(f"  [OK] MCP: {name} - Streamable HTTP active")
        else:
            print(f"  [WARN] MCP: {name} - {err_msg}")

## Step 3: Generate Continue Configuration

Copy the output below to `~/.continue/config.json` (or use the Continue settings UI).

In [ ]:
API_KEY = os.getenv("MAAS_API_KEY", "sk-oai-YOUR-KEY")

continue_config = {
    "models": [
        {
            "title": f"{MODEL_NAME} (RHOAI via MaaS)",
            "provider": "openai",
            "model": MODEL_NAME,
            "apiBase": f"{MAAS_HOST}/v1",
            "apiKey": API_KEY
        }
    ],
    "tabAutocompleteModel": {
        "title": "Autocomplete (RHOAI)",
        "provider": "openai",
        "model": MODEL_NAME,
        "apiBase": f"{MAAS_HOST}/v1",
        "apiKey": API_KEY
    },
    "mcpServers": [
        {"name": name, "url": url}
        for name, url in mcp_servers.items()
    ]
}

print("=" * 60)
print("~/.continue/config.json")
print("=" * 60)
print(json.dumps(continue_config, indent=2))

## Step 4: Install & Configure Continue

### 4a. Install Extension

1. Open VS Code Extensions (`Cmd+Shift+X`)
2. Search **"Continue"** and install
3. Open Continue sidebar panel

![Continue Extension Install](screenshots/07-vscode-continue-install.png)

### 4b. Apply Configuration

Paste the config from Step 3 into `~/.continue/config.json`.

![Continue Config](screenshots/08-vscode-continue-config.png)

## Step 5: Test Continue Features

### 5a. Chat Mode

1. Open Continue chat panel
2. Ask: *"Explain what this file does"* with a file open
3. Verify the model responds with accurate analysis

![Continue Chat](screenshots/09-vscode-continue-chat.png)

### 5b. Inline Edit

1. Select a code block
2. Press `Cmd+I` to trigger inline edit
3. Prompt: *"Add error handling and type hints"*
4. Verify the edit is applied correctly

![Continue Inline Edit](screenshots/10-vscode-continue-inline.png)

### 5c. MCP Tool Calling

1. In chat, ask: *"Use the GitHub tool to check open PRs in this repo"*
2. Verify Continue invokes the MCP tool via MaaS

![Continue Tool Call](screenshots/11-vscode-continue-tool.png)

## Step 6: Verification Summary

In [ ]:
print("VS Code + Continue - Verification Checklist")
print("=" * 60)
print(f"  Model Endpoint : {MAAS_HOST}/v1")
print(f"  Model Name     : {MODEL_NAME}")
print(f"  MCP Servers    : {len(mcp_servers)} configured")
print()
print("  [ ] Continue extension installed")
print("  [ ] config.json applied")
print("  [ ] Chat mode works (model responds)")
print("  [ ] Inline edit works (Cmd+I)")
print("  [ ] MCP tool calling works")
print()
print("Add screenshots to 6_ide_integration_test/screenshots/")

## Troubleshooting

| Issue | Fix |
|-------|-----|
| "Could not connect to server" | Verify `apiBase` URL from Step 1 is reachable |
| Slow responses | Check model pod resource utilization (`oc get pods`) |
| MCP tools not showing | Restart Continue extension after config change |
| SSL errors | Set `"requestOptions": {"verifySsl": false}` in config |